# Tokenizing Your "Verdict Book" — A Student-Friendly, Step-by-Step Notebook

This notebook is designed to be practical and beginner-friendly.

By the end, you will be able to:
1. Load your book from your **Downloads** folder.
2. Clean and normalize the text.
3. Tokenize it in multiple ways (word-level and model-level).
4. Inspect token statistics and chunk text for LLM workflows.
5. Save outputs for later experiments.


## 0) What is tokenization (in plain English)?

**Tokenization** means splitting text into smaller units called **tokens**.

- In classic NLP, tokens are often words or punctuation.
- In modern LLMs, tokens are subword pieces (for example, `tokenization` may split into smaller chunks).

Why this matters:
- Model context limits are measured in **tokens**, not characters.
- Cost and latency often scale with token count.
- Better token awareness = better prompt and chunking strategies.


## 1) Setup

If this is your first run, uncomment the install cell below.


In [ ]:
# Uncomment and run once if needed:
# !pip install -q pypdf tiktoken matplotlib pandas


In [ ]:
from pathlib import Path
import re
import collections

import pandas as pd
import matplotlib.pyplot as plt


## 2) Point to your Verdict Book file in Downloads

Update `BOOK_FILENAME` to match your actual file name.

Examples:
- `Verdict Book.pdf`
- `verdict_book.txt`


In [ ]:
DOWNLOADS_DIR = Path.home() / "Downloads"
BOOK_FILENAME = "Verdict Book.pdf"  # <-- change this if your filename differs
book_path = DOWNLOADS_DIR / BOOK_FILENAME

print("Looking for:", book_path)
print("Exists?", book_path.exists())


### Optional: list likely files in Downloads
If you are unsure of the exact filename, run this:


In [ ]:
candidates = sorted([
    p.name for p in DOWNLOADS_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {".pdf", ".txt", ".docx", ".epub"}
])

print("Possible text/book files:")
for name in candidates[:100]:
    print(" -", name)


## 3) Read text from the file

This notebook supports `.txt` and `.pdf` directly.

- `.txt`: direct read.
- `.pdf`: page-by-page extraction via `pypdf`.

If your book is in another format (`.docx`, `.epub`), convert it to `.txt` or `.pdf` first.


In [ ]:
def load_book_text(path: Path) -> str:
    suffix = path.suffix.lower()

    if suffix == ".txt":
        return path.read_text(encoding="utf-8", errors="replace")

    if suffix == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(str(path))
        pages = []
        for i, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text() or ""
            pages.append(page_text)
            if i % 50 == 0:
                print(f"Extracted page {i}/{len(reader.pages)}")
        return "\n".join(pages)

    raise ValueError(f"Unsupported file type: {suffix}. Use .txt or .pdf")


In [ ]:
raw_text = load_book_text(book_path)
print("Character count:", len(raw_text))
print("Preview:\n", raw_text[:1000])


## 4) Clean and normalize text

A simple cleaner helps remove extra whitespace and normalize weird quote/dash characters.


In [ ]:
def normalize_text(text: str) -> str:
    # Normalize common unicode punctuation
    replacements = {
        "\u2018": "'", "\u2019": "'",  # curly single quotes
        "\u201c": '"', "\u201d": '"',  # curly double quotes
        "\u2013": "-", "\u2014": "-",  # en/em dash
        "\u00a0": " ",                  # non-breaking space
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    # Collapse whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

clean_text = normalize_text(raw_text)
print("Cleaned character count:", len(clean_text))


## 5) Tokenization Method A — Word-level (simple + transparent)

Great for learning and quick analysis.


In [ ]:
word_tokens = re.findall(r"\b\w+\b", clean_text.lower())
print("Word token count:", len(word_tokens))
print("First 40 tokens:", word_tokens[:40])


In [ ]:
freq = collections.Counter(word_tokens)
most_common = freq.most_common(25)

pd.DataFrame(most_common, columns=["token", "count"])


In [ ]:
# Plot top 20 words
top_n = 20
df_top = pd.DataFrame(freq.most_common(top_n), columns=["token", "count"])

plt.figure(figsize=(12, 5))
plt.bar(df_top["token"], df_top["count"])
plt.xticks(rotation=45, ha="right")
plt.title("Top Word Tokens")
plt.tight_layout()
plt.show()


## 6) Tokenization Method B — LLM-style tokenization with `tiktoken`

This is closer to how OpenAI models count tokens.


In [ ]:
import tiktoken

# Good default for modern OpenAI models
encoding = tiktoken.get_encoding("cl100k_base")
llm_tokens = encoding.encode(clean_text)

print("LLM token count:", len(llm_tokens))
print("First 50 token IDs:", llm_tokens[:50])
print("Decoded snippet:", encoding.decode(llm_tokens[:80]))


### Compare word tokens vs LLM tokens


In [ ]:
comparison = pd.DataFrame([
    {"metric": "characters", "value": len(clean_text)},
    {"metric": "word_tokens", "value": len(word_tokens)},
    {"metric": "llm_tokens_cl100k", "value": len(llm_tokens)},
])
comparison


## 7) Chunk the book into token-safe pieces

Chunking is essential for retrieval, summarization, and long-document QA.


In [ ]:
def chunk_by_tokens(text: str, encoding, max_tokens: int = 800, overlap: int = 100):
    token_ids = encoding.encode(text)
    chunks = []
    start = 0

    while start < len(token_ids):
        end = start + max_tokens
        chunk_ids = token_ids[start:end]
        chunks.append(encoding.decode(chunk_ids))
        if end >= len(token_ids):
            break
        start = end - overlap

    return chunks

chunks = chunk_by_tokens(clean_text, encoding, max_tokens=800, overlap=100)
print("Number of chunks:", len(chunks))
print("First chunk preview:\n", chunks[0][:1200])


## 8) Save outputs (recommended)

This lets you reuse results in later notebooks or pipelines.


In [ ]:
out_dir = Path("tokenization_outputs")
out_dir.mkdir(exist_ok=True)

# Save clean text
(out_dir / "verdict_book_clean.txt").write_text(clean_text, encoding="utf-8")

# Save word frequencies
pd.DataFrame(freq.items(), columns=["token", "count"]) \
  .sort_values("count", ascending=False) \
  .to_csv(out_dir / "word_frequencies.csv", index=False)

# Save chunks
for i, ch in enumerate(chunks, start=1):
    (out_dir / f"chunk_{i:04d}.txt").write_text(ch, encoding="utf-8")

print("Saved files in:", out_dir.resolve())


## 9) Mini exercises (to learn deeply)

1. Try `max_tokens=400` and compare chunk count.
2. Remove stopwords and see how top-word stats change.
3. Build chapter-level token counts (if chapter headings exist).
4. Compare token counts before and after cleaning.


## 10) Troubleshooting

- **File not found**: print `DOWNLOADS_DIR` and list candidates.
- **PDF extraction looks messy**: many PDFs are layout-heavy; try converting to `.txt` first.
- **Install issues**: run the install cell manually and restart kernel.
- **Unicode artifacts**: expand `normalize_text()` replacements.

---

If you want, next step is a second notebook: **"Tokenization + Embeddings + Semantic Search"** using the same book.
